# 15.02 · Probabilities, calibration and thresholds

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 3–4 h |
| 📋 Prerequisites | Lesson 15.01, and Lesson 14.03 (ticket router, ECE, `route_curve`). |
| 🔑 Keys | optional `TYPESAFE_API_KEY` (234 small requests, about $0.002) · optional `GROQ_API_KEY` for the LLM comparison |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (no network, free) · **⚖️** Jev: live with `TYPESAFE_API_KEY` (paid, fractions of a cent), otherwise replayed from recordings · **💸** calls Groq (uses your free-tier quota).
> Without keys, ⚖️ cells replay recorded answers, 💸 cells print `⏭️ Skipped`, and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Tell apart** `confidence`, the top probability and the margin, and know which one to set a threshold on.
2. **Measure calibration** on labeled data: reliability table and plot, expected calibration error (ECE), Brier score.
3. **Pick a routing threshold from costs** ("a misroute costs $2.50, an escalation $0.50") on one half of the data, and report it on the other.
4. **Design an uncertain band** for a yes/no answer: act, ignore, or ask a human.
5. **Compare** with an LLM's self-reported confidence, and check self-consistency across re-runs.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [TypeSafe models](https://docs.typesafe.ai/models) | Model IDs and what is billed |
| [TypeSafe integration (LangChain)](https://docs.langchain.com/oss/python/integrations/providers/typesafe) | `ChoiceAnswer.probabilities`, `confidence`, `NoulAnswer.noul` |
| [Jev integrations guide (Vercel)](https://vercel.com/i/jev-integrations) | `confidenceBelow` vs `probabilityBetween` fallbacks; why the threshold must match the question type |
| [On Calibration of Modern Neural Networks (Guo et al., 2017)](https://arxiv.org/abs/1706.04599) | Reliability diagrams and ECE, the standard method |

> **Why this matters at work.** In Lesson 15.01, `ROUTE_AT = 0.6` was a guess. A guessed threshold is either too strict, so you pay for an LLM on tickets Jev had right, or too loose, so misroutes reach customers. Orbita's support lead can tell you what each mistake costs. This lesson turns those costs and 234 labeled tickets into a threshold you can defend, and checks first that the probabilities deserve to be trusted.

> 🧪 **About the replayed lab answers.** Until real Jev answers are recorded, the 234 lab answers are *synthetic*, and **calibrated by construction**: a ticket shown with top probability 0.9 was made right 90% of the time. So in replay mode this lesson proves the **method**: how to measure calibration and how to pick thresholds. It does not prove that Jev is calibrated. With `TYPESAFE_API_KEY` the same cells measure the real model.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-typesafe[experimental]==0.0.1a3" "langchain-groq>=1.1.3,<2" "matplotlib>=3.11.2,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-typesafe", "langchain-groq", "matplotlib",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY')
LANGSMITH_PROJECT = 'm15-calibration-and-thresholds'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.decisions import (BandCosts, apply_band, band_cost, brier_score, choose_band, decision_cost, margin,
                               provenance, top_probability)
from mastery.decisions.orbita import TICKETS, lab_halves, lab_tickets, queue_questions, ticket_state, triage_questions
from mastery.finetune.metrics import best_threshold, calibration, expected_calibration_error, route_curve
from mastery.finetune.predictions import RECORDED_DIR, load_predictions

LAB = lab_tickets()
TUNE, CHECK = lab_halves()
print(f"lab: {len(LAB)} labeled tickets (Module 14's held-out split) → tune {len(TUNE)} / check {len(CHECK)}")

**You should now see** `lab: 234 labeled tickets … → tune 117 / check 117`. These are Module 14's held-out tickets: six queues, some Spanish, some deliberately hard. You choose thresholds on **tune** and report on **check**. Choosing and reporting on the same tickets flatters every threshold.

---
## Part 1 · Three numbers that look alike

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ Replays Lesson 15.01's T5 (the ambiguous ticket).
r = jev.invoke({"state": ticket_state(TICKETS[4]), "questions": triage_questions()})
team = r.choices["team"]
print("probabilities :", dict(sorted(team.probabilities.items(), key=lambda kv: -kv[1])))
print(f"top probability: {top_probability(team):.2f}   ← P(the chosen label is right), if calibrated")
print(f"margin         : {margin(team):.2f}   ← top minus runner-up: 'is there a close second?'")
print(f"confidence     : {team.confidence:.2f}   ← TypeSafe's summary of the distribution's shape")

**You should now see** a top probability of `0.52`, a margin of `0.08` and a confidence of `0.08`. (In replay mode, the synthetic `confidence` is the margin. TypeSafe's real formula is its own, so read it live.)

| Number | Answers | Use it for |
|---|---|---|
| **Top probability** | "How often is a label shown with this probability right?" | **Calibration**: compare it with accuracy |
| **Margin** | "Is there a close runner-up?" | Spotting two-intent tickets (T5: login *and* billing) |
| **`confidence`** | TypeSafe's single number, derived from the whole distribution | A default threshold, *after* you've checked it against accuracy |

TypeSafe's docs say thresholds "should be chosen according to the consequences of" a wrong answer. The rest of this lesson does exactly that, using **top probability**, because it is the one you can check against how often the model is right.

---
## Part 2 · Ask the lab set

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ 234 requests, two questions each (queue = Choice over six queues, is_outage = Noul). Live: ~50K input tokens ≈ $0.002.
start = time.perf_counter()
answers = jev.batch([{"state": t["text"], "questions": queue_questions()} for t in LAB], config={"max_concurrency": 8})
seconds = time.perf_counter() - start

conf = {t["id"]: top_probability(a.choices["queue"]) for t, a in zip(LAB, answers)}
right = {t["id"]: int(a.choices["queue"].choice == t["queue"]) for t, a in zip(LAB, answers)}
p_outage = {t["id"]: a.nouls["is_outage"].noul for t, a in zip(LAB, answers)}
is_outage = {t["id"]: int(t["queue"] == "outage") for t in LAB}
cost = sum(decision_cost(a) or 0 for a in answers)
print(f"{provenance(answers[0])} · {len(answers)} answers in {seconds:.1f} s · ${cost:.5f}")
print(f"queue accuracy: {sum(right.values()) / len(LAB):.1%}  (Module 14's few-shot qwen3:4b on the same tickets: 94.4%)")

**You should now see** `replay (synthetic) · 234 answers …`, a cost of about `$0.002`, and a queue accuracy of about `89%`. Accuracy alone says nothing about **which** answers to trust. That is what calibration is for.

---
## Part 3 · Is the probability honest? (calibration)

In [ ]:
# 🧪 Offline: reliability table, ECE and Brier on the answers above.
ids = [t["id"] for t in LAB]
c, ok = [conf[i] for i in ids], [right[i] for i in ids]
table(calibration(c, ok, bins=5), ["bin", "n", "confidence", "accuracy"])
print(f"\nECE (10 bins)        : {expected_calibration_error(c, ok):.3f}   (0 = perfect; under ~0.05 is good)")
print(f"Brier, is_outage     : {brier_score([p_outage[i] for i in ids], [is_outage[i] for i in ids]):.3f}   "
      f"(0 = perfect; {sum(is_outage.values()) / len(ids) * (1 - sum(is_outage.values()) / len(ids)):.3f} = always guessing the base rate)")

**You should now see** three bins. Most tickets sit in `0.8–1.0` with accuracy close to their confidence (about `0.91` vs `0.94`). The ECE is about `0.04`, and the Brier score is far below the base-rate guess.

- **Reliability table:** group answers by probability and compare each group's mean probability with how often it was right. Calibrated means the two match.
- **ECE:** the size-weighted average of those gaps. **Brier:** the squared error of yes-probabilities, for `Noul` answers.
- **Small bins are noisy.** A bin with 11 tickets can be 25 points off by chance. Look at `n` before you conclude anything, and pool data from several weeks in production.

In [ ]:
# 🧪 Reliability diagram: dots on the diagonal = calibrated.
import matplotlib.pyplot as plt

rows = calibration(c, ok, bins=10)
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.scatter([r["confidence"] for r in rows], [r["accuracy"] for r in rows], s=[12 + 2 * r["n"] for r in rows],
           label=f"Jev ({provenance(answers[0])})")
ax.set(xlabel="top probability", ylabel="accuracy", xlim=(0.3, 1.02), ylim=(0.3, 1.02), title="Reliability, 234 tickets")
ax.legend(loc="lower right", fontsize=8)
plt.show()

**You should now see** a square plot with the dots near the dashed diagonal. Bigger dots hold more tickets. A dot **below** the line is over-confident, and one **above** is under-confident.

---
## Part 4 · A routing threshold from costs

Orbita's support lead puts numbers on the two mistakes:

- A **misrouted** ticket costs about **$2.50**: an agent re-routes it, and the customer waits longer.
- An **escalated** ticket costs about **$0.50**: a bigger model's tokens plus a second of latency. For the escalated tickets we use Module 14's recorded few-shot `qwen3:4b` labels on the same tickets (94% right), so this runs offline.

`route_curve` (Module 14) prices "keep Jev's answer if top probability ≥ t, else escalate" at every threshold. We pick the cheapest on **tune** and report it on **check**.

In [ ]:
# 🧪 Offline: choose on TUNE, report on CHECK.
escalated = load_predictions(RECORDED_DIR / "fewshot-qwen3-4b.json")["labels"]
COST_WRONG, COST_ESCALATE = 2.50, 0.50

def curve(tickets, thresholds=None):
    return route_curve([conf[t["id"]] for t in tickets], [right[t["id"]] for t in tickets],
                       [int(escalated.get(t["id"]) == t["queue"]) for t in tickets],
                       cost_wrong=COST_WRONG, cost_escalate=COST_ESCALATE, thresholds=thresholds)

tune_points = curve(TUNE)
best = best_threshold(tune_points)
table([{"threshold": p.threshold, "kept": p.kept, "escalated": p.escalated, "accuracy": p.accuracy, "$ total": p.cost}
       for p in tune_points if p.threshold in (0.0, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0)],
      ["threshold", "kept", "escalated", "accuracy", "$ total"])
(check,) = curve(CHECK, [best.threshold])
(never,) = curve(CHECK, [0.0])
(always,) = curve(CHECK, [1.01])
print(f"\nchosen on tune: t = {best.threshold}")
print(f"on check: keep {check.kept}, escalate {check.escalated}, accuracy {check.accuracy:.1%}, ${check.cost:.2f} "
      f"| never escalate ${never.cost:.2f} ({never.accuracy:.1%}) | always escalate ${always.cost:.2f} ({always.accuracy:.1%})")

**You should now see** a cost curve that falls and then rises again, with the cheapest threshold around `0.8`. On **check**, that policy escalates about one ticket in nine and is cheaper than both extremes: *never escalate* (more misroutes) and *always escalate* (every ticket pays for the LLM).

- Change `COST_WRONG` and the best threshold moves. **The threshold is a business decision expressed in numbers**, and it is not a property of the model.
- Report the **check** number. The tune number is optimistic by construction.

---
## Part 5 · An uncertain band for a yes/no answer

`is_outage` triggers a page to the on-call engineer. The mistakes are lopsided:

- A **missed outage** costs about **$200**: customers are down and nobody knows.
- A **false page** costs about **$20**: someone gets woken up.
- A **human glance** at the ticket costs about **$1**.

A single cutoff can't serve both directions, so use **two**. Page if `p ≥ high`, ignore if `p < low`, and in between send the ticket to the support lead for review. Vercel's AI Gateway calls the same idea `probabilityBetween`: on a yes/no answer, the *middle* band is the uncertain one, while low probabilities mean a confident "no".

In [ ]:
# 🧪 Offline: pick (low, high) on TUNE, report on CHECK, compare with a single 0.5 cutoff.
COSTS = BandCosts(false_yes=20, false_no=200, review=1)
tp, ty = [p_outage[t["id"]] for t in TUNE], [is_outage[t["id"]] for t in TUNE]
cp, cy = [p_outage[t["id"]] for t in CHECK], [is_outage[t["id"]] for t in CHECK]

band = choose_band(tp, ty, COSTS)
on_check = band_cost(cp, cy, band.low, band.high, COSTS)
single = band_cost(cp, cy, 0.5, 0.5, COSTS)
print(f"chosen on tune : page if p ≥ {band.high}, ignore if p < {band.low}, review in between")
print(f"on check (band): ${on_check.cost:.0f} · {on_check.reviewed} reviewed · {on_check.errors} wrong actions")
print(f"on check (0.5) : ${single.cost:.0f} · 0 reviewed · {single.errors} wrong actions")
for t in CHECK[:40]:
    if apply_band(p_outage[t["id"]], band.low, band.high) == "review":
        print(f"  review: p={p_outage[t['id']]:.2f} gold={t['queue']:<8} {t['text'][:70]}")

**You should now see** a band of roughly `[0.45, 0.55)` or wider, a handful of reviewed tickets on check, and a cost no higher than the single `0.5` cutoff. The reviewed tickets are mostly hard outage-or-bug cases: exactly the ones a person should look at.

The width of the band is your "human budget". If the support lead can review 5 tickets an hour, check that `reviewed` fits before you ship the band.

---
## Part 6 · Compared with an LLM's self-reported confidence

In [ ]:
# 💸 40 tune tickets in 2 Groq calls (~6K tokens): queue + a self-reported confidence, then the same calibration check.
from typing import Literal

from pydantic import BaseModel, Field

from mastery.finetune.llm_router import fewshot_prompt
from mastery.models import get_model, structured_model

Queue = Literal["billing", "outage", "account_access", "bug", "how_to", "feature_request"]

class Labeled(BaseModel):
    id: str
    queue: Queue
    confidence: float = Field(ge=0, le=1, description="Your probability (0-1) that this queue is correct")

class LabeledBatch(BaseModel):
    labels: list[Labeled]

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = structured_model(get_model("fast", temperature=0), LabeledBatch)
    sample, got = TUNE[:40], {}
    for b in range(0, len(sample), 20):
        batch = sample[b:b + 20]
        out = llm.invoke(fewshot_prompt() + "\nAlso give your confidence for each label.\n\n"
                         + "\n".join(f"{t['id']}: {t['text']}" for t in batch))
        got.update({x.id: x for x in out.labels})
    llm_c = [got[t["id"]].confidence if t["id"] in got else 0.0 for t in sample]
    llm_ok = [int(t["id"] in got and got[t["id"]].queue == t["queue"]) for t in sample]
    jev_c, jev_ok = [conf[t["id"]] for t in sample], [right[t["id"]] for t in sample]
    table([{"model": "LLM (self-reported)", "accuracy": f"{sum(llm_ok) / 40:.0%}", "mean conf": f"{sum(llm_c) / 40:.2f}",
            "ECE": expected_calibration_error(llm_c, llm_ok), "distinct values": len(set(llm_c))},
           {"model": f"Jev ({provenance(answers[0])})", "accuracy": f"{sum(jev_ok) / 40:.0%}", "mean conf": f"{sum(jev_c) / 40:.2f}",
            "ECE": expected_calibration_error(jev_c, jev_ok), "distinct values": len(set(round(x, 3) for x in jev_c))}],
          ["model", "accuracy", "mean conf", "ECE", "distinct values"])

**You should now see** (with a Groq key) the LLM's confidences bunched on a few round values (`0.9`, `0.95`), often well above its accuracy on the hard tickets. Jev has many distinct values. Forty tickets is a small sample, so treat the ECE gap as a hint, not a verdict. Without a key the cell prints `⏭️ Skipped`.

An LLM's "confidence" is a number it *writes*. Nothing in its training ties that number to how often it is right. That is the gap decision models are built to close, and Part 3's method is how you check whether one does on *your* data.

In [ ]:
# ⚖️ Self-consistency: ask the first 20 lab tickets again and compare the probabilities.
again = jev.batch([{"state": t["text"], "questions": queue_questions()} for t in LAB[:20]])
drift = max(abs(top_probability(a.choices["queue"]) - conf[t["id"]]) for t, a in zip(LAB[:20], again))
flips = sum(a.choices["queue"].choice != b.choices["queue"].choice for a, b in zip(again, answers[:20]))
print(f"{provenance(again[0])}: max probability change {drift:.4f}, label flips {flips}/20")

**You should now see** `max probability change 0.0000, label flips 0/20`. In replay mode that is trivially true, because the answers are the same recordings. **Live** it is the real test: TypeSafe says Jev is designed to be self-consistent, and LangChain reported judge scores that "barely moved" across 100 runs. Run it live once and see.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Threshold looks great, production is worse | Chosen and reported on the same tickets | Tune/check split (or cross-validation); report check only |
| A bin is 30 points off | Few tickets in it | Look at `n`; use fewer bins; collect more labels |
| Threshold on `confidence` behaves oddly | `confidence` ≠ probability of being right | Calibrate the top probability; use `confidence` only after checking it |
| A Noul "uncertain" rule fires on every "no" | `confidenceBelow`-style rule on a yes/no probability | Use a middle band (`low ≤ p < high`), as in Part 5 |
| `ReplayMiss` after editing `queue_questions()` | New question set, no recording | Expected; go live, or keep the lesson's questions |

## ⚠️ Production notes

> ⚠️ **Production note: re-check calibration on every model change.** (Parts 3–5)
> `jev-latest` moves to new releases. A new release can shift probabilities, and every threshold tuned on the old one with them. Pin the model, keep the labeled set, and re-run Parts 3–5 before you switch.

> ⚠️ **Production note: log probabilities, not just labels.** (Part 4)
> With the probability logged for every decision, plus the eventual human correction, you can re-tune thresholds every month from production data without calling the model again.

## 🏋️ Exercises

1. **(Easy)** Set `COST_WRONG = 10` in Part 4. Where does the best threshold go, and why?
2. **(Medium)** Repeat Part 4 using `a.choices["queue"].confidence` (TypeSafe's confidence) instead of the top probability. Is the cheapest policy different? Which one would you trust, and why?
3. **(Medium)** Calibration per language: compute ECE separately for `lang == "es"` and `"en"`. What changes, and is the Spanish sample large enough to say?
4. **(Stretch, needs a key)** Run Parts 2–5 live and compare the real Jev ECE and the chosen thresholds with the synthetic replay. Record both in `docs/research-log.md`.

<details>
<summary><b>Show solutions</b></summary>

**1.** Misroutes become 4× pricier, so escalating gets relatively cheaper and the best threshold rises (typically `0.9`). You escalate more tickets to avoid costly mistakes.

**2.** Replace `conf` with `{t["id"]: a.choices["queue"].confidence for ...}` and re-run. Confidence is not on the same scale as accuracy, so the chosen value differs even if the ranking of tickets is similar. Trust whichever you have **checked** on held-out data; the top probability is the one calibration can verify directly.

**3.** `es = [t for t in LAB if t["lang"] == "es"]` and the same `expected_calibration_error` on that subset. With about 30 Spanish tickets the estimate is very noisy (a bin of 5 tickets moves 20 points per ticket), so collect more before claiming a language gap.

**4.** Unset nothing; set the key and Restart & Run All. The synthetic answers are calibrated by construction, so a real ECE that is clearly higher is a genuine finding, and your thresholds should come from the live run.
</details>

## 🔎 Docs drill (5 min)

In the Vercel Jev integrations guide, which AI Gateway fallback condition suits a Choice question, which suits a Boolean question, and why?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Jev integrations](https://vercel.com/i/jev-integrations) → *Add conditional fallbacks through AI Gateway*.
2. **`confidenceBelow`** for a Choice question (the CSV column-mapping example).
3. **`probabilityBetween`** for a Boolean question: a *low* probability means a confident "no", so only the **middle band** is uncertain. That is the same reasoning as Part 5.

</details>

## ✅ Checkpoint

- [ ] I can explain top probability vs margin vs `confidence`.
- [ ] I measured a reliability table, ECE and Brier on labeled data, and know what small bins do.
- [ ] I chose a routing threshold from costs on one half and reported it on the other.
- [ ] I built an act / ignore / review band for a yes/no answer.
- [ ] I know why an LLM's self-reported confidence is not a calibrated probability.
- [ ] Restart & Run All passes with no keys; outputs cleared before saving.

## 🔎 Research checkpoint

Search TypeSafe's docs for how `confidence` is computed from the distribution. If it is documented, note the formula in [`docs/research-log.md`](../../docs/research-log.md). If it isn't, note that, and keep calibrating on the top probability.

## Summary

- A probability is useful only if it is **calibrated**. Measure it with a reliability table, ECE (Choice) and Brier (Noul) on labeled data.
- **Thresholds come from costs**, chosen on one half of the data and reported on the other (`route_curve`, `best_threshold`).
- For yes/no automations, use an **uncertain band** (`choose_band`): act, ignore, or review.
- An LLM's self-reported confidence is generated text. Check it the same way before you use it.
- Re-check calibration and thresholds whenever the model version changes.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 15-03-jev-in-langchain.ipynb** → [`15-03-jev-in-langchain.ipynb`](15-03-jev-in-langchain.ipynb).

[Module 15 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)